# IHDP Smoke Test for MOCA One-Way Cut

This notebook is an example for IHDP dataset.  It will load IHDP, fit, predict CATE/ATE, and save a small result table for 5 different seeds.


## 1. Imports and Package Path

Run this notebook from either the repository root or the `moca-oneway-cut` package directory. If the package is not installed yet, the local `src` folder is added to `sys.path`.


In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

cwd = Path.cwd()
package_root = cwd if (cwd / "src" / "moca_oneway_cut").exists() else cwd / "moca-oneway-cut"
if not (package_root / "src" / "moca_oneway_cut").exists():
    package_root = Path(__file__).resolve().parents[1] if "__file__" in globals() else cwd.parent

src_dir = package_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from moca_oneway_cut import MOCAOneWayCuttingFeedback

outputs_dir = package_root / "outputs"
outputs_dir.mkdir(exist_ok=True)

#print(f"Package root: {package_root}")
#print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


## 2. Load IHDP

By default this downloads one IHDP replication from the public CEVAE repository. To use a local file instead, set `IHDP_CSV_PATH` before running this cell.


In [2]:
def load_ihdp_data(rep=1, csv_path=None):
    if csv_path is None:
        csv_path = os.environ.get("IHDP_CSV_PATH")
    if csv_path is None:
        csv_path = (
            "https://raw.githubusercontent.com/AMLab-Amsterdam/CEVAE/"
            f"master/datasets/IHDP/csv/ihdp_npci_{rep}.csv"
        )

    colnames = ["treatment", "y_factual", "y_cfactual", "mu0", "mu1"] + [f"X{i}" for i in range(1, 26)]
    df = pd.read_csv(csv_path, header=None, names=colnames)
    df["X14"] = df["X14"] - 1

    x_cols = [f"X{i}" for i in range(1, 26)]
    X = df[x_cols].to_numpy(dtype=np.float32)
    T = df["treatment"].to_numpy(dtype=np.float32)
    Y = df["y_factual"].to_numpy(dtype=np.float32)
    mu0 = df["mu0"].to_numpy(dtype=np.float32)
    mu1 = df["mu1"].to_numpy(dtype=np.float32)
    tau = mu1 - mu0

    return {
        "source": str(csv_path),
        "frame": df,
        "X": X,
        "T": T,
        "Y": Y,
        "mu0": mu0,
        "mu1": mu1,
        "tau": tau,
    }


data = load_ihdp_data(rep=1)
#print(data["source"])
#print(data["frame"].shape)
#data["frame"].head()


## 3. Seeds and Split Helper
 
 The split follows `IHDP_analysis.ipynb`: 60% train, 20% validation, 20% test, stratified by treatment. The next cells run five independent seeds.


In [3]:
seeds = [2022, 2023, 2024, 2025, 2026]
idx = np.arange(data["X"].shape[0])
 
def make_split(seed):
    train_idx, tmp_idx = train_test_split(
        idx,
        test_size=0.4,
        random_state=seed,
        stratify=data["T"],
    )
    valid_idx, test_idx = train_test_split(
        tmp_idx,
        test_size=0.5,
        random_state=seed,
        stratify=data["T"][tmp_idx],
    )
    return train_idx, valid_idx, test_idx

#print({"seeds": seeds, "n_seeds": len(seeds)})


## 4. Fit and Evaluate MOCA Across Seeds
 
 This cell uses the same high-level training settings as `IHDP_analysis.ipynb`. Each seed gets a fresh stratified split and a fresh model initialization.


In [4]:
rows = []
device = "cuda" if torch.cuda.is_available() else "cpu"

for seed in seeds:
    print(f"\n=== Seed {seed} ===")
    train_idx, valid_idx, test_idx = make_split(seed)

    X_train, T_train, Y_train = data["X"][train_idx], data["T"][train_idx], data["Y"][train_idx]
    X_val, T_val, Y_val = data["X"][valid_idx], data["T"][valid_idx], data["Y"][valid_idx]
    X_test, T_test, Y_test = data["X"][test_idx], data["T"][test_idx], data["Y"][test_idx]
    tau_test = data["tau"][test_idx]

    est = MOCAOneWayCuttingFeedback(
        d_model=32,
        nhead=4,
        num_layers=1,
        dropout=0.1,
        gate_temp=1.0,
        batch_size=128,
        treat_epochs=40,
        outcome_epochs=60,
        lr_treat=1e-3,
        lr_outcome=3e-4,
        validation_fraction=0.0,
        random_state=seed,
        device=device,
        verbose=True,
    )
    est.fit(X_train, T_train, Y_train, X_val=X_val, T_val=T_val, Y_val=Y_val)

    tau_hat = est.effect(X_test)
    ate_hat = est.ate(X_test)
    factual_hat = est.predict(X_test, treatment=T_test)
    propensity_hat = est.propensity(X_test)

    ate_true = float(tau_test.mean())
    ate_bias = float(ate_hat - ate_true)
    cate_rmse = float(np.sqrt(np.mean((tau_hat - tau_test) ** 2)))
    ate_abs_error = float(abs(ate_bias))
    factual_mse = float(np.mean((factual_hat - Y_test) ** 2))

    rows.append({
        "dataset": "IHDP",
        "rep": 1,
        "seed": seed,
        "n_train": len(train_idx),
        "n_valid": len(valid_idx),
        "n_test": len(test_idx),
        "method": "MOCAOneWayCuttingFeedback",
        "treat_epochs": est.config.treat_epochs,
        "outcome_epochs": est.config.outcome_epochs,
        "ate_true": ate_true,
        "ate_hat": float(ate_hat),
        "ate_bias": ate_bias,
        "ate_abs_error": ate_abs_error,
        "cate_rmse": cate_rmse,
        "factual_mse": factual_mse,
        "propensity_mean": float(propensity_hat.mean()),
    })

results = pd.DataFrame(rows)
results



=== Seed 2022 ===
[MOCA-Treat] Epoch 001 | train_loss=0.6460 | valid_loss=0.6259
[MOCA-Treat] Epoch 002 | train_loss=0.6253 | valid_loss=0.5945
[MOCA-Treat] Epoch 003 | train_loss=0.5915 | valid_loss=0.5462
[MOCA-Treat] Epoch 004 | train_loss=0.5496 | valid_loss=0.4756
[MOCA-Treat] Epoch 005 | train_loss=0.4947 | valid_loss=0.4237
[MOCA-Treat] Epoch 006 | train_loss=0.4988 | valid_loss=0.4248
[MOCA-Treat] Epoch 007 | train_loss=0.4959 | valid_loss=0.4235
[MOCA-Treat] Epoch 008 | train_loss=0.4837 | valid_loss=0.4353
[MOCA-Treat] Epoch 009 | train_loss=0.4912 | valid_loss=0.4389
[MOCA-Treat] Epoch 010 | train_loss=0.4969 | valid_loss=0.4327
[MOCA-Treat] Epoch 011 | train_loss=0.4738 | valid_loss=0.4253
[MOCA-Treat] Epoch 012 | train_loss=0.4879 | valid_loss=0.4184
[MOCA-Treat] Epoch 013 | train_loss=0.4825 | valid_loss=0.4165
[MOCA-Treat] Epoch 014 | train_loss=0.4722 | valid_loss=0.4183
[MOCA-Treat] Epoch 015 | train_loss=0.4647 | valid_loss=0.4181
[MOCA-Treat] Epoch 016 | train_loss=

,dataset,rep,seed,n_train,n_valid,n_test,method,treat_epochs,outcome_epochs,ate_true,ate_hat,ate_bias,ate_abs_error,cate_rmse,factual_mse,propensity_mean
0,IHDP,1,2022,448,149,150,MOCAOneWayCuttingFeedback,40,60,4.028909,3.993996,-0.034913,0.034913,0.820446,1.962846,0.157996
1,IHDP,1,2023,448,149,150,MOCAOneWayCuttingFeedback,40,60,4.054395,4.261552,0.207157,0.207157,0.481722,1.228954,0.150906
2,IHDP,1,2024,448,149,150,MOCAOneWayCuttingFeedback,40,60,4.088966,4.096834,0.007868,0.007868,0.469275,1.312341,0.207545
3,IHDP,1,2025,448,149,150,MOCAOneWayCuttingFeedback,40,60,4.049766,4.017119,-0.032647,0.032647,0.547900,1.448794,0.188653
4,IHDP,1,2026,448,149,150,MOCAOneWayCuttingFeedback,40,60,3.955269,4.045010,0.089741,0.089741,0.926886,2.378612,0.190650


## 5. Save Raw Results and Summary
 
 This cell saves one row per seed and a compact mean/std summary for the main metrics.


In [5]:
metric_cols = ["ate_bias", "ate_abs_error", "cate_rmse", "factual_mse", "propensity_mean"]
summary = results[metric_cols].agg(["mean", "std"]).T.reset_index().rename(columns={"index": "metric"})

raw_output_path = outputs_dir / "ihdp_smoke_test_results_5seeds.csv"
summary_output_path = outputs_dir / "ihdp_smoke_test_summary_5seeds.csv"
results.to_csv(raw_output_path, index=False)
summary.to_csv(summary_output_path, index=False)

#print(f"Saved raw results: {raw_output_path}")
#print(f"Saved summary: {summary_output_path}")
summary


,metric,mean,std
0,ate_bias,0.047441,0.102544
1,ate_abs_error,0.074465,0.079982
2,cate_rmse,0.649246,0.210427
3,factual_mse,1.666310,0.489692
4,propensity_mean,0.179150,0.023844
